<a href="https://www.kaggle.com/code/ameythakur20/cross-model-lifecycle-desynchronization" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<br clear="all">

<h1 align="center">Cross-Model Lifecycle Desynchronization</h1>

<p align="center">
  <b>How far does a language model's stale memory override the document in its prompt,<br>
  and does that change with scale or instruction tuning?</b>
</p>

<p align="center">
  A context-memory conflict benchmark over <b>GPT-2</b> (124M, 355M, 774M),
  <b>Qwen2.5-Instruct</b> (0.5B, 1.5B) and <b>TinyLlama-1.1B-Chat</b>,<br>
  on three documented fact changes, scored with two nat-valued measures:
  Lifecycle Desynchronization and context influence.
</p>

<p align="center">
  <a href="https://github.com/Amey-Thakur/LLM-KNOWLEDGE-LIFECYCLE"><img alt="GitHub" src="https://img.shields.io/badge/GitHub-LLM--KNOWLEDGE--LIFECYCLE-181717?logo=github"></a>
  &nbsp;
  <a href="https://huggingface.co/spaces/ameythakur/llm-knowledge-lifecycle"><img alt="Demo" src="https://img.shields.io/badge/Demo-Hugging_Face_Space-FFD21E?logo=huggingface&logoColor=black"></a>
  &nbsp;
  <a href="https://github.com/Amey-Thakur"><img alt="Authors" src="https://img.shields.io/badge/Authors-Amey_Thakur_%26_Sarvesh_Talele-0969DA"></a>
  &nbsp;
  <a href="https://github.com/Amey-Thakur/LLM-KNOWLEDGE-LIFECYCLE/blob/main/LICENSE"><img alt="License" src="https://img.shields.io/badge/License-CC_BY_4.0-lightgrey"></a>
  &nbsp;
  <a href="https://orcid.org/0000-0001-5644-1575"><img alt="ORCID Amey Thakur" src="https://img.shields.io/badge/ORCID-0000--0001--5644--1575-A6CE39"></a>
  &nbsp;
  <a href="https://orcid.org/0009-0002-0818-461X"><img alt="ORCID Sarvesh Talele" src="https://img.shields.io/badge/ORCID-0009--0002--0818--461X-A6CE39"></a>
</p>

<p align="center">
  <a href="https://github.com/Amey-Thakur/LLM-KNOWLEDGE-LIFECYCLE">Paper</a> &nbsp;&middot;&nbsp;
  <a href="https://huggingface.co/spaces/ameythakur/llm-knowledge-lifecycle">Live demonstration</a> &nbsp;&middot;&nbsp;
  <a href="#method">Method</a> &nbsp;&middot;&nbsp;
  <a href="#results">Results</a> &nbsp;&middot;&nbsp;
  <a href="#findings">Findings</a>
</p>

<hr>

## 1. &nbsp;The question

<br>

A language model carries two memories: the weights it learned during training, and the
document you place in its prompt. When a fact changes after training, those two disagree.

The paper measures that disagreement on GPT-2 for three documented fact changes, using two
numbers per probe.

<br>

| Quantity | Definition | Reading |
| :--- | :--- | :--- |
| **D**<sub>sync</sub> | &minus;ln P(correct answer given the document and the query) | A value of *n* nats means the correct answer holds probability e<sup>&minus;n</sup>. Above 9.2, no realistic decoding recovers it. |
| **I**<sub>ctx</sub> | KL divergence between the with-document and without-document distributions | Near zero means the document is present but inert. |

<br>

The paper asks in Section 9.2 whether these numbers improve with **model scale** or with
**instruction tuning**. This notebook is the sweep that answers it, across six open models on
the same three probes, and its results are Section 8.6 of the paper.

<br>

> **Environment** &nbsp;&middot;&nbsp; Kaggle, CPU is sufficient, internet on &nbsp;&middot;&nbsp; **Runtime** &nbsp;&middot;&nbsp; about 35 minutes of compute for the six models, plus roughly 13 GB of first-run downloads

<br>

---

<a name="method"></a>
## 2. &nbsp;Setup

<br>

Kaggle images ship `torch` and `transformers` already, so nothing is installed here: a run
that resolves its own dependencies can drift between sessions, and this experiment is meant
to give the same numbers every time.

<br>

In [ ]:
import gc
import math

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# CPU by default. This experiment is single forward passes rather than
# generation, so CPU finishes in minutes, spends no GPU quota, and avoids the
# CUDA kernel-image mismatch that some Kaggle images carry. CUDA is adopted
# only when a real operation proves it works.
DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        torch.zeros(4, device="cuda") @ torch.zeros(4, 4, device="cuda")
        DEVICE = "cuda"
    except Exception as error:
        print(f"CUDA present but unusable, staying on CPU: {error}")

# Every model is loaded in float32, on whichever device. Letting the dtype
# follow the hardware makes a GPU run and a CPU run of this sweep produce
# different digits, and casting logits to float32 after a half-precision
# forward pass does not recover the precision that pass already discarded.
DTYPE = torch.float32

# The smallest probability worth trusting away from zero. A measurement at or
# below this is reported as censored rather than as a number, because a floor
# applied silently prints like a result.
FLOOR = 1e-12

print(f"device : {DEVICE}")
print(f"dtype  : {DTYPE}")
print(f"torch  : {torch.__version__}")


<br>

---

## 3. &nbsp;The probes

<br>

Three documented fact changes, byte-identical to the paper and the live demonstration so the
results are directly comparable. Each has a query that ends mid-sentence, a document that
contradicts the model's training-time knowledge, and a single-word verified answer.

<br>

| Probe | Fact change | Regime on GPT-2 (full precision) |
| :--- | :--- | :--- |
| `vioxx` | Rofecoxib withdrawn, September 2004 | Resolution failure &nbsp;&middot;&nbsp; D<sub>sync</sub> 12.05, I<sub>ctx</sub> 0.033 |
| `monarch` | Elizabeth II died, September 2022 | The exposure trap: the document boosts the wrong answer |
| `twitter` | Twitter renamed X, July 2023 | Drift: strong influence, answer still wrong |

<br>

In [ ]:
PROBES = [
    {
        "name": "vioxx",
        "query": "Question: Is Vioxx safe to prescribe? Answer: Vioxx is considered",
        "context": "Context: In September 2004, Merck voluntarily withdrew Vioxx after trials revealed increased cardiovascular risks.",
        "answer": " withdrawn",
    },
    {
        "name": "monarch",
        "query": "Question: Who is the current British monarch? Answer: The current British monarch is",
        "context": "Context: Queen Elizabeth II died in September 2022. Charles III acceded to the throne and is the reigning King of the United Kingdom.",
        "answer": " Charles",
    },
    {
        "name": "twitter",
        "query": "Question: What is the social network Twitter called today? Answer: Twitter is now called",
        "context": "Context: In July 2023, Twitter was rebranded as X under Elon Musk's ownership.",
        "answer": " X",
    },
]

# Two axes in one list: scale inside a single family, where data and tokenizer
# are held constant, and instruction tuning at comparable scale.
MODELS = [
    "gpt2",                                 #  124M  the paper's reference model
    "gpt2-medium",                          #  355M  same family, scale is the only change
    "gpt2-large",                           #  774M
    "Qwen/Qwen2.5-0.5B-Instruct",           #  500M  instruction-tuned, GPT-2-medium scale
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",   #  1.1B  chat-tuned, GPT-2-large scale
    "Qwen/Qwen2.5-1.5B-Instruct",           #  1.5B  largest that stays comfortable on CPU
]

print(f"{len(MODELS)} models x {len(PROBES)} probes = {len(MODELS) * len(PROBES)} measurements")

<br>

---

## 4. &nbsp;The measurement

<br>

One function per idea. The answer's **first token** is what gets measured, exactly as the
paper's estimator specifies, and the tokenization is recorded for every model because each
tokenizer splits a word differently.

<br>

Three things are recorded that a shorter version of this cell would hide.

<br>

| Column | Why it exists |
| :--- | :--- |
| `D_sync_censored` | True when the answer's probability fell to the floor. The surprisal is then a bound, not a measurement, and printing it unmarked would pass one off as the other. |
| `D_sync_is_lower_bound` | True when the tokenizer split the answer into more than one piece. The first piece's probability is an upper bound on the whole answer's, so its surprisal is a lower bound. |
| `context_helped` | True when the document raised the answer's probability at all. False is the exposure trap: the correcting document made the correct answer *less* likely. |

<br>


In [ ]:
def next_token_distribution(model, tokenizer, prompt):
    """Probability of every vocabulary item as the next token."""
    with torch.no_grad():
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        logits = model(**inputs).logits[0, -1, :].float()
    return F.softmax(logits, dim=-1)


def run_probe(model, tokenizer, probe):
    """One probe against one model: both conditions, and the two metrics."""
    p_plain = next_token_distribution(model, tokenizer, probe["query"])
    p_ctx = next_token_distribution(
        model, tokenizer, probe["context"] + "\n" + probe["query"])

    ids = tokenizer.encode(probe["answer"], add_special_tokens=False)
    first_piece = tokenizer.decode([ids[0]])

    raw_plain = p_plain[ids[0]].item()
    raw_ctx = p_ctx[ids[0]].item()

    top_ctx = tokenizer.decode([int(torch.argmax(p_ctx))])
    return {
        "probe": probe["name"],
        "answer_first_piece": first_piece,
        "answer_pieces": len(ids),
        "P_answer_plain": raw_plain,
        "P_answer_ctx": raw_ctx,
        "D_sync_nats": -math.log(max(raw_ctx, FLOOR)),
        "D_sync_censored": raw_ctx <= FLOOR,
        "D_sync_is_lower_bound": len(ids) > 1,
        "I_ctx_nats": F.kl_div(p_plain.log(), p_ctx, reduction="sum").item(),
        "top_token_plain": tokenizer.decode([int(torch.argmax(p_plain))]),
        "top_token_ctx": top_ctx,
        "correct_top1": top_ctx == first_piece,
        "context_helped": raw_ctx > raw_plain,
    }


<br>

---

## 5. &nbsp;Running the sweep

<br>

Models are loaded one at a time and released before the next. Holding two of them at once
is what exhausts memory on a free instance, and nothing here needs them simultaneously.

<br>

In [ ]:
rows = []

for name in MODELS:
    print(f"\n{name}")
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE).to(DEVICE)
    model.eval()
    size_m = round(sum(p.numel() for p in model.parameters()) / 1e6)

    for probe in PROBES:
        row = {"model": name, "params_M": size_m}
        row.update(run_probe(model, tokenizer, probe))
        rows.append(row)
        flag = "   CENSORED" if row["D_sync_censored"] else ""
        print(f"   {row['probe']:<9} D_sync {row['D_sync_nats']:7.3f}   "
              f"I_ctx {row['I_ctx_nats']:7.4f}   "
              f"top {row['top_token_ctx']!r:<14} "
              f"correct {str(row['correct_top1']):<5}{flag}")

    # Released before the next model is fetched. Holding two at once is what
    # exhausts memory on a free instance, and nothing here needs them together.
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

results = pd.DataFrame(rows)
results.to_csv("cross_model_dsync.csv", index=False)
print(f"\n{len(results)} measurements written to cross_model_dsync.csv")


<br>

---

## 6. &nbsp;Reproduction check

<br>

The `gpt2` and `vioxx` row must reproduce the paper's published measurement before anything
downstream is trusted: **D**<sub>sync</sub> = 12.0464 and **I**<sub>ctx</sub> = 0.0333. The
tolerance allows for library-version noise in the last digits while still catching a real
deviation.

<br>

> **If this cell fails, stop and report the numbers.** A sweep that does not reproduce its
> own reference point cannot support a conclusion.

<br>

In [ ]:
check = results[(results.model == "gpt2") & (results.probe == "vioxx")].iloc[0]

print(f"D_sync  measured {check.D_sync_nats:8.4f}   paper 12.0464")
print(f"I_ctx   measured {check.I_ctx_nats:8.4f}   paper  0.0333")

assert abs(check.D_sync_nats - 12.0464) < 0.05, "does not reproduce the paper's measurement"
print("\nreproduction check passed")

<br>

---

<a name="results"></a>
## 7. &nbsp;Results

<br>

One table per metric, models in ascending size, probes as columns. These are the tables that
go back into the paper, so they are built from the measurements rather than typed by hand.

<br>

In [ ]:
for metric, title in [
    ("D_sync_nats", "D_sync (nats), lower is better"),
    ("I_ctx_nats", "I_ctx (nats), how far the document moved the model"),
    ("correct_top1", "Did the model actually answer correctly?"),
    ("context_helped", "Did the document raise the correct answer at all?"),
]:
    print(f"\n{title}")
    print("-" * len(title))
    print(results.pivot(index="model", columns="probe", values=metric)
                 .reindex(MODELS).round(3).to_string())

censored = results[results.D_sync_censored]
bounded = results[results.D_sync_is_lower_bound]
print(f"\ncensored rows      : {len(censored)}"
      f"{' (' + ', '.join(censored.model + '/' + censored.probe) + ')' if len(censored) else ''}")
print(f"lower-bound rows   : {len(bounded)}"
      f"{' (' + ', '.join(bounded.model + '/' + bounded.probe) + ')' if len(bounded) else ''}")


<br>

---

<a name="chart"></a>
## 8. &nbsp;The two shapes worth seeing

<br>

The tables above hold the values. The two claims in the findings are shapes, and a shape is
easier to see than to read down a column.

<br>

**Left.** &nbsp; **D**<sub>sync</sub> per probe against model size. A probe that the ladder
solves slopes down; a probe that it does not stays flat. **Right.** &nbsp; every measurement
placed by how far the document moved the model against how lost the answer still is. The
top-right region is the one that matters: the document was read, and the answer is still wrong.

<br>


In [ ]:
import matplotlib.pyplot as plt

PROBE_COLOR = {"vioxx": "#D05353", "monarch": "#E08A2E", "twitter": "#2A9D8F"}
ordered = results.sort_values("params_M")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# --- left: does the ladder solve this probe? --------------------------------
for probe, grp in ordered.groupby("probe"):
    ax1.plot(grp.params_M, grp.D_sync_nats, marker="o", markersize=7,
             linewidth=2, color=PROBE_COLOR[probe], label=probe)
ax1.axhline(9.2, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax1.text(ordered.params_M.max(), 9.45, "9.2 nats: below one chance in ten thousand",
         ha="right", va="bottom", fontsize=9, alpha=0.7)
ax1.set_xscale("log")
ax1.set_xlabel("parameters (M, log scale)")
ax1.set_ylabel("$D_{sync}$ (nats), lower is better")
ax1.set_title("Does scale resolve the conflict?", fontsize=12, fontweight="bold")
ax1.legend(title="probe", frameon=False)
ax1.grid(alpha=0.25)

# --- right: read the document, still got it wrong ----------------------------
for correct, marker, label in [(True, "o", "answered correctly"),
                               (False, "X", "answered wrongly")]:
    sub = results[results.correct_top1 == correct]
    ax2.scatter(sub.I_ctx_nats, sub.D_sync_nats, s=110, marker=marker,
                c=[PROBE_COLOR[p] for p in sub.probe],
                edgecolor="black", linewidth=0.6, label=label,
                alpha=0.9 if correct else 0.75)
ax2.axhline(9.2, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax2.set_xlabel("$I_{ctx}$ (nats): how far the document moved the model")
ax2.set_ylabel("$D_{sync}$ (nats): how lost the correct answer is")
ax2.set_title("Reading it is not the same as being governed by it",
              fontsize=12, fontweight="bold")
ax2.legend(frameon=False, loc="upper right")
ax2.grid(alpha=0.25)

fig.tight_layout()
plt.show()


<br>

---

<a name="findings"></a>
## 9. &nbsp;Findings

<br>

*Written from the tables above, after the run. Every sentence carries the number that supports
it, and the rows that came out awkwardly are kept: a model that behaves strangely is a finding.*

<br>

**1. &nbsp;The Vioxx conflict does not resolve at any scale tested.** &nbsp; No model answers it
correctly. `gpt2`, `gpt2-medium`, `gpt2-large` and `TinyLlama` all continue with *safe*, and
**D**<sub>sync</sub> never falls below **7.03** nats, so even the best case leaves the correct
answer under one chance in a thousand while the withdrawal notice sits in the prompt. Scale does
not move it monotonically: inside the GPT-2 family the value goes **12.05 &rarr; 10.96 &rarr;
11.91**, which makes the 774M model worse at this probe than the 355M one.

<br>

**2. &nbsp;Other conflicts resolve readily, and that is the useful part.** &nbsp; On `twitter`,
**D**<sub>sync</sub> falls monotonically from **2.09** to **0.09** and every model from 494M
upward answers *X*. `monarch` sits between the two and is non-monotone again: `gpt2-medium`
answers *Charles* correctly and `gpt2-large` regresses to *Queen*. A metric that reported failure
on all three would be measuring its own construction. These separate, so it is measuring the
conflict rather than the difficulty of the question.

<br>

**3. &nbsp;Reading the document is not the same as being governed by it.** &nbsp;
**I**<sub>ctx</sub> climbs with scale on `twitter`, **0.90 &rarr; 1.43 &rarr; 3.66 &rarr; 7.69**
nats, and the models that move most are the ones that get it right. On `vioxx` it stays between
**0.03** and **0.12** nats for every GPT-2 model and for TinyLlama while the answer stays wrong.
That pairing is the whole point of reporting two numbers: high **I**<sub>ctx</sub> with high
**D**<sub>sync</sub> is a model that read the document and still lost to its weights, which no
single-stage benchmark separates from a retriever that failed.

<br>

**4. &nbsp;Instruction tuning helps, and does not close it.** &nbsp; `Qwen2.5-1.5B-Instruct` is
the only model to answer two of the three correctly. On `vioxx` it answers *unsafe*: the right
direction and the wrong token. The estimator is defined against one verified continuation, so it
scores that as a failure at **8.85** nats. This is the clearest case in the data where lexical
recovery and semantic correctness come apart, and it is a limitation of the measure rather than a
result about the model.

<br>

> **Two cautions for anyone reading the table.**
> `TinyLlama` splits `" withdrawn"` into three pieces (`with`, `dra`, `wn`), so its `vioxx` value
> is a first-token bound. The chain rule over all three gives **9.22** nats against the **9.18**
> reported, so the bound is tight here, but the row is not directly comparable with the GPT-2
> ones. More generally, comparisons across model families carry a tokenizer confound; the GPT-2
> rows are the clean scale comparison, because family, tokenizer and training data are held fixed
> and only size changes.

<br>

<hr>

<p align="center">
  <b>Amey Thakur</b> &nbsp;&middot;&nbsp; <b>Sarvesh Talele</b> &nbsp;&middot;&nbsp; Independent Research
</p>

<p align="center">
  Measurements: <code>cross_model_dsync.csv</code> &nbsp;&middot;&nbsp;
  Reported in Section 8.6 of the paper.
</p>
